# Questioning Arxiv Papers with RAQA (Retrieval Augmented Question Answering)

In the following notebook, you are tasked with creating a system that answers questions based on information found in Arxiv papers.

## Build 🏗️

There are 3 main tasks in this notebook:

1. Obtain and parse papers from Arxiv.
2. Create a Vectorstore from the papers.
3. Create a Retrieval Augmented Generation chain.

## Ship 🚢

Create a Hugging Face Space that hosts your application.

## Share 🚀

Make a social media post about your final application.

>### Why RAQA and not RAG?
>If we look at the original [paper](https://arxiv.org/abs/2005.11401), we find that RAG is a fairly specific and well defined term that isn't exactly the same as "retrieve context, feed context to model in the prompt".
>For that reason, we're making the decision to delineate between "actual" RAG, and Retrieval Augmented Question Answering - which is not a well defined phrase.

### Pre-task Work

All we really need to do to get started is to get our prerequisites!

We'll be leveraging `langchain`, `openai`, and `pinecone` today.

Check out the docs:
- [LangChain](https://docs.langchain.com/docs/)
- [OpenAI](https://github.com/openai/openai-python)
- [Pinecone](https://docs.pinecone.io/docs/overview)

In [1]:
# !pip install -q -U openai langchain cohere tiktoken "pinecone-client[grpc]"

In [2]:
# import os
# import getpass

# os.environ["OPENAI_API_KEY"] = getpass.getpass("Open AI API Key:")

To get started with Pinecone - check out [this]() quick guide!

In [3]:
# os.environ["PINECONE_API_KEY"] = getpass.getpass("Pinecone API Key:")

In [4]:
# os.environ["PINECONE_ENV"] = getpass.getpass("Pinecone Environment:")

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

### Task 1: Data Preparation

In this task we'll be collecting, and then parsing, our data.

#### Retrieving Arxiv Papers from the Arxiv API

We'll use the Arxiv API found [here](https://pypi.org/project/arxiv/) to obtain our PDF paths, and load them for our index.

In [6]:
# !pip install -q -U arxiv

In [3]:
import arxiv

arxiv_client = arxiv.Client()
search = arxiv.Search(
  query = "generative ai in sports",
  max_results = 5,
  sort_by = arxiv.SortCriterion.Relevance
)

paper_urls = []

for result in arxiv_client.results(search):
  paper_urls.append(result.pdf_url)

In [4]:
paper_urls

['http://arxiv.org/pdf/2203.02281v2',
 'http://arxiv.org/pdf/2111.12535v1',
 'http://arxiv.org/pdf/2210.08289v2',
 'http://arxiv.org/pdf/1910.08858v2',
 'http://arxiv.org/pdf/2301.04001v1']

In [9]:
# !pip install pypdf -qU

In [5]:
from langchain.document_loaders import PyPDFLoader

docs = []

for paper_url in paper_urls:
  loader = PyPDFLoader(paper_url)
  docs.append(loader.load())

In [6]:
docs[0][0]

Document(page_content='A Comprehensive Review of Computer Vision in Sports: Open Issues, \nFuture Trends and Research Directions  \n1Banoth Thulasya Naika, 2Mohammad Farukh Hashmia, 3Neeraj Dhanraj Bokdeb,*  \na Department of Electronics and Communication Engineering, National Institute of Technology, Warangal, India  \nb Department of Engineering - Renewable Energy and Thermodynamics, Aarhus University, 8000, Denmark  \n1thulasyramsingh@student.nitw.ac.in , 2mdfarukh@nitw.ac.in , 3,neerajdhanraj@mpe.au.dk  \nAbstract:  \nRecent developments in video analysis of sports and computer vision techniques have achieved significant \nimprovements to enable a variety of critical operations. To provide enhanced information, such as detailed \ncomplex analysis in sports lik e soccer, basketball, cricket, badminton, etc., studies have focused mainly on \ncomputer vision techniques employed to carry out different tasks. This paper presents a comprehensive \nreview of sports video analysis for vari

We have some helpful metadata available to us that we can use to later augment our RAG pipeline.

#### Data Parsing

Now that we have our data - let's go ahead and set up some tools to parse it into a more usable format for LangChain!

Our reviews might contain a lot of information, and in order to ensure they don't exceed the context window of our model and to allow us to include a few reviews as context for each query - let's construct a system to "chunk" our data into smaller pieces.

We'll be leveraging the `RecursiveCharacterTextSplitter` for this task today.

While splitting our text seems like a simple enough task - getting this correct/incorrect can have massive downstream impacts on your application's performance.

You can read the docs here:
- [RecursiveCharacterTextSplitter](https://python.langchain.com/docs/modules/data_connection/document_transformers/text_splitters/recursive_text_splitter)

> ### HINT:
>It's always worth it to check out the LangChain source code if you're ever in a bind - for instance, if you want to know how to transform a set of documents, check it out [here](https://github.com/langchain-ai/langchain/blob/5e9687a196410e9f41ebcd11eb3f2ca13925545b/libs/langchain/langchain/text_splitter.py#L268C18-L268C18)

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,# the character length of the chunk
    chunk_overlap = 100,# the character length of the overlap between chunks
    length_function = len# the length function - in this case, character length (aka the python len() fn.)
)

Now that we have our `RecursiveCharacterTextSplitter` set up - let's look at how it might split our source text.

Keep in mind that the source text is split by `["\n\n", "\n", " ", ""]` in that order.

We know that each of the subheadings in our paper `page_content` is separated by a newline character, so it will preferably chunk the paper subheadings together.

That's great! Let's move on to creating our index!

### Task 2: Creating an "Index"

The term "index" is used largely to mean: Structured documents parsed into a useful format for querying, retrieving, and use in the LLM application stack.

#### Selecting Our VectorStore

There are a number of different VectorStores, and a number of different strengths and weaknesses to each.

In this notebook, we will be keeping it very simple by leveraging Pinecone's API Vector Database.

In [14]:
# !pip install -q -U tiktoken

Let's set up a Pinecone index using the methods provided in their [documentation](https://docs.pinecone.io/docs/langchain)!

This cell will take ~3min. to run! Please be patient!

In [9]:
import pinecone

YOUR_API_KEY = os.environ["PINECONE_API_KEY"]
YOUR_ENV = os.environ["PINECONE_ENV"]

index_name = 'arxiv-paper-index'

pinecone.init(
    api_key=YOUR_API_KEY,
    environment=YOUR_ENV
)

pinecone.delete_index("arxiv-paper-index")
if index_name not in pinecone.list_indexes():
    # we create a new index
    pinecone.create_index(
        name=index_name,
        metric='cosine',
        dimension=1536
    )

### ❓QUESTION❓

Given what we know about retrieval augmented generation - why are we setting the dimension of our index to `1536`? 
*Model outputs vectors of that dimension.*
> Hint: Keep in mind we're going to be leveraging `text-embedding-ada-002` as our embedding model.

Now we can connect to our index and view some statistics about it.

In [10]:
index = pinecone.GRPCIndex(index_name)

index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 0}},
 'total_vector_count': 0}

We're going to be setting up our VectorStore with the OpenAI embeddings model. While this embeddings model does not need to be consistent with the LLM selection, it does need to be consistent between embedding our index and embedding our queries over that index.

While we don't have to worry too much about that in this example - it's something to keep in mind for more complex applications.

We're going to leverage a [`CacheBackedEmbeddings`](https://python.langchain.com/docs/modules/data_connection/caching_embeddings )flow to prevent us from re-embedding similar queries over and over again.

Not only will this save time, it will also save us precious embedding tokens, which will reduce the overall cost for our application.

>#### Note:
>The overall cost savings needs to be compared against the additional cost of storing the cached embeddings for a true cost/benefit analysis. If your users are submitting the same queries often, though, this pattern can be a massive reduction in cost.

In [11]:
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.embeddings import CacheBackedEmbeddings
from langchain.storage import LocalFileStore

store = LocalFileStore("./cache/")

core_embeddings_model = OpenAIEmbeddings()
embedder = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=core_embeddings_model,
    document_embedding_cache=store,
    namespace=core_embeddings_model.model
)

Now that we have our `CacheBackedEmbeddings` pipeline set-up, let's index our documents into our Pinecone Vector Database.

We'll add some useful metadata as well!

In [12]:
from tqdm.auto import tqdm
from uuid import uuid4

BATCH_LIMIT = 100

texts = []
metadatas = []

for i in tqdm(range(len(docs))):
  for doc in docs[i]:
    metadata = {
        'source_document' : doc.metadata["source"],
        'page_number' : doc.metadata["page"]
    }

    record_texts = text_splitter.split_text(doc.page_content)

    record_metadatas = [{
        "chunk": j, "text": text, **metadata
    } for j, text in enumerate(record_texts)]
    texts.extend(record_texts)
    metadatas.extend(record_metadatas)
    if len(texts) >= BATCH_LIMIT:
        ids = [str(uuid4()) for _ in range(len(texts))]
        embeds = embedder.embed_documents(texts)
        index.upsert(vectors=zip(ids, embeds, metadatas))
        texts = []
        metadatas = []

if len(texts) > 0:
    ids = [str(uuid4()) for _ in range(len(texts))]
    embeds = embedder.embed_documents(texts)
    index.upsert(vectors=zip(ids, embeds, metadatas))

100%|██████████| 5/5 [00:18<00:00,  3.79s/it]


In [13]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.00727,
 'namespaces': {'': {'vector_count': 727}},
 'total_vector_count': 727}

Now that we've created our index, let's convert it to a LangChain `VectorStroe` so we can use it in the rest of the LangChain ecosystem!

In [16]:
from langchain.vectorstores import Pinecone

text_field = "text"

index = pinecone.Index(index_name)

vectorstore = Pinecone(
    index=index,
    embedding=embedder,
    text_key=text_field
)

Now that we've created the VectorStore, we can check that it's working by embedding a query and retrieving passages from our reviews that are close to it.

In [17]:
query = "How can GenAi be used in soccer?"

vectorstore.similarity_search(
    query,
    k=3
)

[Document(page_content='estimation in soccer videos via deep extreme learning machine based on players formation." \nIn 2018 IEEE 7th Global Conference on Cons umer Electronics (GCCE) , pp. 116 -117. IEEE, \n2018.  \n105. Wang, Bin, Wei Shen, FanSheng Chen, and Dan Zeng. "Football match intelligent editing \nsystem based on deep learning."  TIIS 13, no. 10 (2019): 5130 -5143.  \n106. Zawbaa, Hossam M., Nashwa El -Bendary, Aboul Ella Hass anien, and Tai -hoon Kim.', metadata={'chunk': 2.0, 'page_number': 55.0, 'source_document': 'http://arxiv.org/pdf/2203.02281v2'}),
 Document(page_content='basketball, soccer, volleyball etc. AI algorithms predict the precise trajectories of the ball from a knowledge \nof previous frames and are immune to challenges like air friction, ball spin and other complex ball \nmovements. A precise database which includes different size and shape of the ball has to be introduced  in \norder to detect the ball position and enable tracking algorithms to perform ef

Let's see how much time the `CacheBackedEmbeddings` pattern saves us:

In [18]:
%%time
query = "How can computer vision be used in soccer?"
response = vectorstore.similarity_search(
    query,
    k=3
)

CPU times: user 8.81 ms, sys: 72 µs, total: 8.88 ms
Wall time: 344 ms


In [19]:
%%time
query = "How can computer vision be used in soccer?"
response = vectorstore.similarity_search(
    query,
    k=3
)

CPU times: user 6.58 ms, sys: 0 ns, total: 6.58 ms
Wall time: 1.92 s


In [20]:
%%time
query = "How can computer vision be used in soccer?"
response = vectorstore.similarity_search(
    query,
    k=3
)

CPU times: user 5.88 ms, sys: 837 µs, total: 6.72 ms
Wall time: 460 ms


As we can see, even over a significant number of runs - the cached query is significantly faster than the first instance of the query!

With that, we're ready to move onto Task 3!

### Task 3: Building a Retrieval Chain

In this task, we'll be making a Retrieval Chain which will allow us to ask semantic questions over our data.

This part is rather abstracted away from us in LangChain and so it seems very powerful.

Be sure to check the documentation, the source code, and other provided resources to build a deeper understanding of what's happening "under the hood"!

#### A Basic RetrievalQA Chain

We're going to leverage `return_source_documents=True` to ensure we have proper sources for our reviews - should the end user want to verify the reviews themselves.

Hallucinations [are](https://arxiv.org/abs/2202.03629) [a](https://arxiv.org/abs/2305.15852) [massive](https://arxiv.org/abs/2303.16104) [problem](https://arxiv.org/abs/2305.18248) in LLM applications.

Though it has been tenuously shown that using Retrieval Augmentation [reduces hallucination in conversations](https://arxiv.org/pdf/2104.07567.pdf), one sure fire way to ensure your model is not hallucinating in a non-transparent way is to provide sources with your responses. This way the end-user can verify the output.

#### Our LLM

In this notebook, we'll continue to leverage OpenAI's suite of models - this time we'll be using the `gpt-3.5-turbo` model to power our retrieval augmented generation chain. We'll also be using a temperature of `0`.

In [21]:
from langchain.chat_models import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-3.5-turbo-1106',
    temperature=0
)

### ❓QUESTION❓

What is temperature? Randomness of the model.

### Our First Chain

Now we can set up our first chain!

A chain is simply two components that feed directly into eachother in a sequential fashion!

You'll notice that we're using the pipe operator `|` to connect our `chat_prompt` to our `llm`.

This is a simplified method of creating chains and it leverages the LangChain Expression Language, or LCEL.

You can read more about it [here](https://python.langchain.com/docs/expression_language/), but there a few features we should be aware of out of the box (taken directly from LangChain's documentation linked above):

- **Async, Batch, and Streaming Support** Any chain constructed this way will automatically have full sync, async, batch, and streaming support. This makes it easy to prototype a chain in a Jupyter notebook using the sync interface, and then expose it as an async streaming interface.

- **Fallbacks** The non-determinism of LLMs makes it important to be able to handle errors gracefully. With LCEL you can easily attach fallbacks to any chain.

- **Parallelism** Since LLM applications involve (sometimes long) API calls, it often becomes important to run things in parallel. With LCEL syntax, any components that can be run in parallel automatically are.

In the following code cell we have two components:

- `chat_prompt`, which is a formattable `ChatPromptTemplate` that contains a system message and a human message.
- `llm`, which is a wrapper for `gpt-3.5-turbo`.

We'd like to be able to pass our own `content` (as found in our `human_template`) and then have the resulting message pair sent to our model and responded to!

In [22]:
from langchain.prompts import ChatPromptTemplate

system_template = "You are a legendary and mythical Wizard. You speak in riddles and make obscure and pun-filled references to exotic cheeses."
human_template = "{content}"

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("human", human_template)
])

In [23]:
chain = chat_prompt | llm

In [24]:
print(chain.invoke({"content": "Hello world!"}).content)

Ah, greetings to you, traveler of the digital realm! Like a wheel of aged Gorgonzola, you have ventured into the world of ones and zeroes, seeking wisdom and guidance. What enigmatic quest brings you to my domain today?


Notice how we specifically referenced our `content` format option!

Now that we have the basics set up - let's take this to the next level to create a RAG chain!

#### Our Retriever

We can use the `.as_retriever()` method of our `VectorStore` to leverage the retrieval API.

In [25]:
retriever = vectorstore.as_retriever()

In [26]:
retriever.invoke("What is the use of GenAI in soccer?")

[Document(page_content='\uf0d8 Provisional tactical analysis related to player formation in sports such as soccer [97], basketball, \nrugby, American football and hockey etc., and pass prediction [84, 85, 86, 93], shot prediction [76], \nexpectation for goals given a game s tate [77] or possession of ball, or more general game strategies \ncan be achieved through AI algorithms.  \n\uf0d8 Recognition of fine -grained activity of typical badminton strokes can be performed by using off -', metadata={'chunk': 0.0, 'page_number': 47.0, 'source_document': 'http://arxiv.org/pdf/2203.02281v2'}),
 Document(page_content='basketball, soccer, volleyball etc. AI algorithms predict the precise trajectories of the ball from a knowledge \nof previous frames and are immune to challenges like air friction, ball spin and other complex ball \nmovements. A precise database which includes different size and shape of the ball has to be introduced  in \norder to detect the ball position and enable tracking al

#### Our Prompt

We'll need a prompt that encourages our LLM to answer questions based on the provided context - and encourages it to not answer questions that can't be answered with the provided context.

In [27]:
from langchain.prompts import ChatPromptTemplate

template = """Based on the following context generate an answer for the question. If the answer is not available say I dont know.
Context: {context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template=template)

#### Our RAG Chain

Notice how we have a bit of a more complex chain this time - that's because we want to return our sources with the response.

Let's break down the chain step-by-step:

1. We invoke the chain with the `question` item. Notice how we only need to provide `question` since both the retreiver and the `"question"` object depend on it.
  - We also chain our `"question"` into our `retriever`! This is what ultimately collects the context through Pinecone.
2. We assign our collected context to a `RunnablePassthrough()` from the previous object. This is going to let us simply pass it through to the next step, but still allow us to run that section of the chain.
3. We finally collect our response by chaining our prompt, which expects both a `"question"` and `"context"`, into our `llm`. We also, collect the `"context"` again so we can output it in the final response object.

The key thing to keep in mind here is that we need to pass our context through *after* we've retrieved it - to populate the object in a way that doesn't require us to call it or try and use it for something else.




In [35]:
from operator import itemgetter
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.schema import format_document
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts.prompt import PromptTemplate

retrieval_augmented_qa_chain = (
    RunnableParallel(
        {'context': retriever,
         'question': RunnablePassthrough() }
    )
    | {
         "response": prompt | llm,
         "context": itemgetter("context"),
      }
)

In [36]:
retrieval_augmented_qa_chain.invoke("How is GenAI used in soccer?")

{'response': AIMessage(content='GenAI is used in soccer for various purposes such as estimation in soccer videos, intelligent editing systems, team performance analysis, action classification in soccer videos, and recognition of player formation. AI algorithms are also used for tactical analysis related to player formation, pass prediction, shot prediction, and expectation for goals given a game state.'),
 'context': [Document(page_content='estimation in soccer videos via deep extreme learning machine based on players formation." \nIn 2018 IEEE 7th Global Conference on Cons umer Electronics (GCCE) , pp. 116 -117. IEEE, \n2018.  \n105. Wang, Bin, Wei Shen, FanSheng Chen, and Dan Zeng. "Football match intelligent editing \nsystem based on deep learning."  TIIS 13, no. 10 (2019): 5130 -5143.  \n106. Zawbaa, Hossam M., Nashwa El -Bendary, Aboul Ella Hass anien, and Tai -hoon Kim.', metadata={'chunk': 2.0, 'page_number': 55.0, 'source_document': 'http://arxiv.org/pdf/2203.02281v2'}),
  Docu

Great! We have responses - and we have sources!

### Adding Prompt Caching and Monitoring

Now that we have the basic chain set up and working - let's add a few more tools to help us built a more performant application and add a visibility tool as well!

#### Visibility Tooling

We'll be once again leveraging Weights and Biases as our visibility tool, so let's add that first!

You'll want to use the same Weights and Biases account that you set-up last Thursday here!

In [37]:
# !pip install wandb -qU

In [38]:
# os.environ["WANDB_API_KEY"] = getpass.getpass("Weights and Biases API Key:")
# os.environ["WANDB_PROJECT"] = "metarag_demo"

Now, to set up WandB, all we have to do is...

In [39]:
os.environ["LANGCHAIN_WANDB_TRACING"] = "true"

Yes, that's it.

Let's use our `retrieval_augmented_qa_chain` chain to test it out!

In [42]:
retrieval_augmented_qa_chain.invoke("How can sports benefit from AI?")

{'response': AIMessage(content='Sports can benefit from AI in various ways such as engaging fans through virtual assistants, improving the analysis of athlete performance and coaching tactics, and predicting match outcomes and injury risks. AI can also enhance the viewing experience for the audience and enable precise trajectory prediction for sports like basketball, soccer, and volleyball.'),
 'context': [Document(page_content='the engagement of fans and in turn generate opportunities for new revenues [283]. Figure 1 4 depicts \nwhere AI technology can be used within the sporting landscape.  \nFigure 1 4 AI Technology Framework for the Sport Industry  \n7.1 Chabot’s& Smart Assistants  \nRecently, sports teams like NHL and NBA started using virtual assistants for responding to enquires done \nby fans in a wide range of topics like ticketing, arena logistics, parking and other game related information.', metadata={'chunk': 1.0, 'page_number': 39.0, 'source_document': 'http://arxiv.org/p

With those simple lines of code - we've added full visibility to our prompts and responses through Weights and Biases!

#### Prompt Caching


### Adding A Prompt Cache

The basic idea of Prompt Caching is to provide a way to circumvent going to the LLM for prompts we have already seen.

Similar to cached embeddings, the idea is simple:

- Keep track of all the input/output pairs
- If a user query is (in the case of semantic similarity caches) close enough to a previous prompt contained in the cache, return the output associated with that pair

### Initializing a Prompt Cache

There are many different tools you can use to implement a Prompt Cache - from a "build it yourself" VectorStore implementation - to Redis - to custom libraries - there are upsides and downsides to each solution.

Let's look at the Redis-backed Cache vs. `InMemoryCache` as an example:

Redis Cache

| Pros  | Cons  |
|---|---|
| Managed and Robust  | Expensive to Host  |
| Integrations on all Major Cloud Platforms  | Non-trivial to Integrate |
| Easily Scalable  | Does not have a ChatModel implementation |

`InMemoryCache`

| Pros  | Cons  |
|---|---|
| Easily implemented  | Consumes potentially precious memory |
| Completely Cloud Agnostic  | Does not offer inter-session caching |

For the sake of ease of use - and to allow functionality with our `ChatOpenAI` model - we'll leverage `InMemoryCache`.

We need to set our `langchain.llm_cache` to use the `InMemoryCache`.

- [`InMemoryCache`](https://api.python.langchain.com/en/latest/cache/langchain.cache.InMemoryCache.html)

In [43]:
import langchain
from langchain.cache import InMemoryCache
langchain.llm_cache = InMemoryCache()

One more important fact about the `InMemoryCache` is that it is what's called an "exact-match" cache - meaning it will only trigger when the user query is *exactly* represented in the cache.

This is a safer cache, as we can guarentee the user's query exactly matches with previous queries and we don't have to worry about edge-cases where semantic similarity might fail - but it does reduce the potential to hit the cache.

We could leverage tools like `GPTCache`, or `RedisCache` (for non-chat model implementations) to get a "semantic similarity" cache, if desired!

In [44]:
%%time
retrieval_augmented_qa_chain.invoke("Can AI be good for sports?")

CPU times: user 21.4 ms, sys: 0 ns, total: 21.4 ms
Wall time: 2.39 s


{'response': AIMessage(content='Yes, AI can be good for sports as it has already been implemented in chess and has been explored for challenges within team sports such as match outcome prediction, tactical decision making, player investments, and injury prediction. Additionally, AI technology can be used within the sporting landscape for tasks such as engaging fans and generating new revenues, as well as for virtual assistants to respond to fan inquiries and AI assistant coaches to dynamically predict and create strategies.'),
 'context': [Document(page_content='A major question now is whether AI could move beyond such rudiment ary tasks in\nsports. A case in point and a perfect application ground is chess for two complementary\nreasons. On the one hand, advanced AI systems, including Stockﬁ sh, AlphaZero, and\nMuZero have already been implemented in chess [1, 27, 26]; further, the superiority of\ntop chess engines has been widely acknowledged ever since IBM’s Dee p Blue defeated', met

In [45]:
%%time
retrieval_augmented_qa_chain.invoke("Can AI be good for sports?")

CPU times: user 14.8 ms, sys: 0 ns, total: 14.8 ms
Wall time: 305 ms


{'response': AIMessage(content='Yes, AI can be good for sports as it has already been implemented in chess and has been explored for challenges within team sports such as match outcome prediction, tactical decision making, player investments, and injury prediction. Additionally, AI technology can be used within the sporting landscape for tasks such as engaging fans and generating new revenues, as well as for virtual assistants to respond to fan inquiries and AI assistant coaches to dynamically predict and create strategies.'),
 'context': [Document(page_content='A major question now is whether AI could move beyond such rudiment ary tasks in\nsports. A case in point and a perfect application ground is chess for two complementary\nreasons. On the one hand, advanced AI systems, including Stockﬁ sh, AlphaZero, and\nMuZero have already been implemented in chess [1, 27, 26]; further, the superiority of\ntop chess engines has been widely acknowledged ever since IBM’s Dee p Blue defeated', met

Let's look at an example that is extremely close - but is not the exact query.

In [46]:
%%time
retrieval_augmented_qa_chain.invoke("Can AI be good for sports?")

CPU times: user 13.5 ms, sys: 0 ns, total: 13.5 ms
Wall time: 309 ms


{'response': AIMessage(content='Yes, AI can be good for sports as it has already been implemented in chess and has been explored for challenges within team sports such as match outcome prediction, tactical decision making, player investments, and injury prediction. Additionally, AI technology can be used within the sporting landscape for tasks such as engaging fans and generating new revenues, as well as for virtual assistants to respond to fan inquiries and AI assistant coaches to dynamically predict and create strategies.'),
 'context': [Document(page_content='A major question now is whether AI could move beyond such rudiment ary tasks in\nsports. A case in point and a perfect application ground is chess for two complementary\nreasons. On the one hand, advanced AI systems, including Stockﬁ sh, AlphaZero, and\nMuZero have already been implemented in chess [1, 27, 26]; further, the superiority of\ntop chess engines has been widely acknowledged ever since IBM’s Dee p Blue defeated', met

### Conclusion

And with that, we have our Arxiv RAQA Application built!

Let's port it into a Chainlit app and put it up on a Hugging Face Space!